# Explicabilité : Oracle 9-nœuds (seed 42) vs V6′ — prédictions + DAG

**But** : comparer l'explicabilité des deux modèles causaux, vite, avec un maximum de graphes.

3 niveaux (chacun indépendant — si un échoue les autres tournent) :
- **Niveau A — Explicabilité structurelle du DAG** : ne charge AUCUN modèle. Lit les matrices `A_dag` sauvegardées + reconstruit `G_phys`. ~15 graphes. Marche même en local.
- **Niveau B — Prédictions (depuis les caches)** : cartes cible/prédiction/erreur, spectres, extrêmes. Pas de sampling → rapide.
- **Niveau C — (optionnel) Sampling + ablation A_dag** : recharge les stacks, échantillonne, ablate A_dag. Colab + checkpoints requis.

> ⚠️ Cadrage honnête : le DAG de ces modèles est **imposé/verrouillé sur le prior physique** (poids ~uniformes), pas découvert. Ces graphes montrent une **structure fidèle et auditable**, pas un mécanisme causal à fort levier.


In [1]:
# === Niveau 0 : setup & chemins ===
import os, sys, json, math
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO_DIR = "/content/climate_data"
    if not os.path.isdir(REPO_DIR):
        os.system("git clone https://github.com/leonelkenfack/climate_data.git " + REPO_DIR)
    os.chdir(REPO_DIR)
    os.system("pip -q install networkx seaborn omegaconf >/dev/null 2>&1")
    DRIVE_ROOT = Path("/content/drive/MyDrive/climate_data")
else:
    REPO_DIR = os.getcwd()
    DRIVE_ROOT = Path(os.environ.get("CLIMATE_DRIVE_ROOT", REPO_DIR))  # local: fallback repo

for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path: sys.path.insert(0, _p)
print("REPO_DIR =", REPO_DIR, "| Colab =", IN_COLAB)

# --- Chemins artefacts ---
RESULTS_9   = Path("results/9node-seed42")                       # local: A_dag_final dans results.json
CKPT_9      = DRIVE_ROOT / "oracle_9node/seed_42/epoch_last.pth" # Colab
CACHE_9     = DRIVE_ROOT / "oracle_9node/seed_42/stage1_cache.pt"
CKPT_V6     = DRIVE_ROOT / "oracle_v6_prime/seed_42/v6_prime_seed42.pth"
CKPT_V6_S1  = DRIVE_ROOT / "oracle_v6_prime/seed_42/stage1_seed42.pth"
CACHE_V6    = DRIVE_ROOT / "oracle_v6_prime/seed_42/bs32b_cache_seed42.pt"

FIG_DIR = Path("results/explainability_report"); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Figures ->", FIG_DIR.resolve())

import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_context("notebook"); HAS_SNS=True
except Exception: HAS_SNS=False
try:
    import networkx as nx; HAS_NX=True
except Exception: HAS_NX=False
try:
    import torch; HAS_TORCH=True
except Exception: HAS_TORCH=False
plt.rcParams["figure.dpi"]=110
print("seaborn:",HAS_SNS,"| networkx:",HAS_NX,"| torch:",HAS_TORCH)


REPO_DIR = C:\Users\reall\Desktop\climate_data\path_c_plus\scripts | Colab = False
Figures -> C:\Users\reall\Desktop\climate_data\path_c_plus\scripts\results\explainability_report


C:\Users\reall\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


seaborn: True | networkx: True | torch: True


In [2]:
# === Niveau 0 : labels des nœuds + G_phys (prior physique) ===
try:
    from st_cdgm.training.physics_prior import (
        VAR_LABELS_9NODE, EXPECTED_EDGES_9NODE,
        VAR_LABELS_V6,    EXPECTED_EDGES_V6,
        build_physical_mask,
    )
    def gphys(nv, labels, edges):
        G = build_physical_mask(num_vars=nv, var_labels=labels, expected_edges=edges)
        return np.asarray(G.detach().cpu().numpy() if hasattr(G,"detach") else G, dtype=float)
    G9  = gphys(9,  VAR_LABELS_9NODE, EXPECTED_EDGES_9NODE)
    G11 = gphys(11, VAR_LABELS_V6,    EXPECTED_EDGES_V6)
    print("[ok] G_phys importe depuis physics_prior")
except Exception as e:
    print("[fallback] physics_prior indisponible:", e)
    VAR_LABELS_9NODE = ["GP850_spat","GP850->GP500","GP500_spat","GP500->GP250","GP250_spat","Q850","W500","IVT","SP_HR"]
    VAR_LABELS_V6    = ["GP850_spat","GP850->GP500","GP500_spat","GP500->GP250","GP250_spat","Q850","W500","IVT","U850","V850","SP_HR"]
    # (src_idx, tgt_idx, sign)
    E9  = [(4,2,1),(2,0,1),(0,8,1),(1,2,1),(3,4,1),(0,5,1),(2,6,1),(5,7,1),(6,8,1),(7,8,1)]
    E11 = [(4,2,1),(2,0,1),(0,10,1),(1,2,1),(3,4,1),(0,5,1),(2,6,1),(5,7,1),(6,10,1),(7,10,1),
           (8,7,1),(9,7,-1),(8,10,1),(9,10,-1)]
    def _mk(nv,edges):
        G=np.zeros((nv,nv));
        for s,t,sg in edges: G[s,t]=sg
        return G
    G9,G11=_mk(9,E9),_mk(11,E11)

print("9-node labels :", VAR_LABELS_9NODE)
print("V6 labels     :", VAR_LABELS_V6)
print("G9 edges:", int((G9!=0).sum()), "| G11 edges:", int((G11!=0).sum()))


C:\Users\reall\Anaconda3\Lib\site-packages\diffusers\models\transformers\transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
C:\Users\reall\Anaconda3\Lib\site-packages\diffusers\models\transformers\transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


[ok] G_phys importe depuis physics_prior
9-node labels : ['GP850_spat', 'GP850->GP500', 'GP500_spat', 'GP500->GP250', 'GP250_spat', 'Q850', 'W500', 'IVT', 'SP_HR']
V6 labels     : ['GP850_spat', 'GP850->GP500', 'GP500_spat', 'GP500->GP250', 'GP250_spat', 'Q850', 'W500', 'IVT', 'U850', 'V850', 'SP_HR']
G9 edges: 10 | G11 edges: 14


In [3]:
# === Niveau 0 : charger A_dag des deux modeles ===
def mask_diag(A):
    A=np.array(A,dtype=float); np.fill_diagonal(A,0.0); return A

def load_9node_Adag():
    # 1) local results.json (rapide, marche hors Colab)
    rj = RESULTS_9/"results.json"
    if rj.exists():
        d=json.load(open(rj));
        if d.get("A_dag_final") is not None:
            print("[9node] A_dag_final <- results.json"); return mask_diag(np.array(d["A_dag_final"]))
    # 2) checkpoint Drive
    if HAS_TORCH and CKPT_9.exists():
        ck=torch.load(CKPT_9,map_location="cpu",weights_only=False)
        sd=ck.get("rcn_cell_state_dict",{})
        if "A_dag" in sd:
            print("[9node] A_dag <- checkpoint"); return mask_diag(sd["A_dag"].cpu().numpy())
    print("[9node] A_dag INTROUVABLE"); return None

def load_v6_Adag():
    for p in (CKPT_V6, CKPT_V6_S1):
        if HAS_TORCH and p.exists():
            ck=torch.load(p,map_location="cpu",weights_only=False)
            if ck.get("A_dag_final") is not None:
                print(f"[v6'] A_dag_final <- {p.name}"); return mask_diag(np.array(ck["A_dag_final"]))
            sd=ck.get("rcn_cell_state_dict",{})
            if "A_dag" in sd:
                print(f"[v6'] A_dag <- {p.name} state_dict"); return mask_diag(sd["A_dag"].cpu().numpy())
    print("[v6'] A_dag INTROUVABLE (checkpoint Drive requis)"); return None

A9  = load_9node_Adag()
A11 = load_v6_Adag()

# metriques 9-node pre-calculees (si dispo)
QJSON9 = {}
qp = RESULTS_9/"q_phys_metrics.json"
if qp.exists(): QJSON9=json.load(open(qp))
print("A9 shape:", None if A9 is None else A9.shape, "| A11 shape:", None if A11 is None else A11.shape)
print("9-node q_phys JSON:", {k:QJSON9.get(k) for k in ["q_phys_binary","q_phys_continuous","skeleton_f1","n_extra_edges","phys_mag_gained"]})


[9node] A_dag INTROUVABLE
[v6'] A_dag INTROUVABLE (checkpoint Drive requis)
A9 shape: None | A11 shape: None
9-node q_phys JSON: {'q_phys_binary': None, 'q_phys_continuous': None, 'skeleton_f1': None, 'n_extra_edges': None, 'phys_mag_gained': None}


In [4]:
# === Niveau 0 : helpers metriques + classification d'aretes ===
try:
    from path_c_plus.scripts.option_c_helpers import (
        compute_q_phys_binary, compute_q_phys_continuous, compute_skeleton_f1)
    HAS_QHELP=True
except Exception as e:
    HAS_QHELP=False; print("[warn] option_c_helpers indispo:", e)

def q_metrics(A, G):
    """Retourne dict de metriques (utilise les helpers officiels si dispo, sinon fallback)."""
    A=mask_diag(A); G=mask_diag(G); out={}
    if HAS_QHELP:
        try:
            qb,_,_,nx_ = compute_q_phys_binary(A,G); out["q_phys_binary"]=float(qb); out["n_extra"]=int(nx_)
        except Exception: pass
        try:
            qc,coll = compute_q_phys_continuous(A,G); out["q_phys_cont"]=float(qc); out["collapsed"]=bool(coll)
        except Exception: pass
        try:
            _f1 = compute_skeleton_f1(A,G); _f1 = _f1[0] if isinstance(_f1,(tuple,list)) else _f1
            out["skeleton_f1"]=float(_f1)
        except Exception: pass
    # fallback simple
    thr=0.05; Ab=(np.abs(A)>thr); Gb=(np.abs(G)>0)
    tp=int((Ab&Gb).sum()); fp=int((Ab&~Gb).sum()); fn=int((~Ab&Gb).sum())
    out.setdefault("skeleton_f1", (2*tp/(2*tp+fp+fn)) if (2*tp+fp+fn)>0 else 0.0)
    out.setdefault("n_extra", fp)
    # accord de signe sur les aretes physiques
    sign_ok=int(((np.sign(A)==np.sign(G))&Gb).sum()); out["sign_correct"]=sign_ok; out["n_phys"]=int(Gb.sum())
    return out

def edge_classes(A, G, thr=0.05):
    """Matrice de classes : 1=correct, -1=signe inverse, 0.5=extra, -0.5=manquant, 0=rien."""
    A=mask_diag(A); G=mask_diag(G); q=A.shape[0]; C=np.zeros((q,q))
    Ab=np.abs(A)>thr; Gb=np.abs(G)>0
    for i in range(q):
        for j in range(q):
            if Gb[i,j] and Ab[i,j]: C[i,j]= 1.0 if np.sign(A[i,j])==np.sign(G[i,j]) else -1.0
            elif Ab[i,j] and not Gb[i,j]: C[i,j]=0.5
            elif Gb[i,j] and not Ab[i,j]: C[i,j]=-0.5
    return C

M9  = q_metrics(A9,  G9)  if A9  is not None else None
M11 = q_metrics(A11, G11) if A11 is not None else None
print("9-node :", M9)
print("V6'    :", M11)


[warn] option_c_helpers indispo: No module named 'path_c_plus'
9-node : None
V6'    : None


In [5]:
# === Niveau A — G1 : heatmaps A_dag appris + G_phys ===
def heat(ax, M, labels, title, vsym=True, cmap="RdBu_r"):
    M=mask_diag(M); v=np.abs(M).max() or 1.0
    im=ax.imshow(M, cmap=cmap, vmin=-v if vsym else 0, vmax=v)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels,rotation=90,fontsize=7); ax.set_yticklabels(labels,fontsize=7)
    ax.set_title(title,fontsize=10); plt.colorbar(im,ax=ax,fraction=0.046)

panels=[]
if A9 is not None: panels+=[(A9,VAR_LABELS_9NODE,"A_dag appris — 9-node"),(G9,VAR_LABELS_9NODE,"G_phys — 9-node")]
if A11 is not None: panels+=[(A11,VAR_LABELS_V6,"A_dag appris — V6'"),(G11,VAR_LABELS_V6,"G_phys — V6'")]
if panels:
    n=len(panels); fig,axes=plt.subplots(1,n,figsize=(5.2*n,4.6)); axes=np.atleast_1d(axes)
    for ax,(M,L,T) in zip(axes,panels): heat(ax,M,L,T)
    fig.suptitle("G1 — Matrices d'adjacence causale (appris vs physique)",y=1.03,fontsize=12)
    plt.tight_layout(); plt.savefig(FIG_DIR/"G1_heatmaps_Adag.png",bbox_inches="tight"); plt.show()


In [6]:
# === Niveau A — G2 : confusion appris vs physique (vert=correct, rouge=signe, orange=extra) ===
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap_c=ListedColormap(["#c0392b","#e67e22","#f7f7f7","#2ecc71"])  # -1,-0.5/0.5,0,1  (approx)
def plot_conf(ax,A,G,labels,title):
    C=edge_classes(A,G)
    # map -1 -> rouge, -0.5 manquant -> rouge clair, 0.5 extra -> orange, 1 correct -> vert
    disp=np.zeros_like(C); disp[C==1]=3; disp[C==0.5]=2; disp[C==-0.5]=1; disp[C==-1]=0; disp[C==0]=1.5
    im=ax.imshow(disp,cmap=ListedColormap(["#c0392b","#f6c1c1","#f7f7f7","#e67e22","#2ecc71"]),vmin=0,vmax=4)
    ax.set_xticks(range(len(labels)));ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels,rotation=90,fontsize=7);ax.set_yticklabels(labels,fontsize=7)
    ax.set_title(title,fontsize=10)

confs=[]
if A9 is not None: confs.append((A9,G9,VAR_LABELS_9NODE,f"9-node — skelF1={M9.get('skeleton_f1',0):.2f} n_extra={M9.get('n_extra','?')}"))
if A11 is not None: confs.append((A11,G11,VAR_LABELS_V6,f"V6' — skelF1={M11.get('skeleton_f1',0):.2f} n_extra={M11.get('n_extra','?')}"))
if confs:
    n=len(confs); fig,axes=plt.subplots(1,n,figsize=(5.4*n,4.8)); axes=np.atleast_1d(axes)
    for ax,(A,G,L,T) in zip(axes,confs): plot_conf(ax,A,G,L,T)
    import matplotlib.patches as mp
    leg=[mp.Patch(color="#2ecc71",label="correct"),mp.Patch(color="#c0392b",label="signe inverse"),
         mp.Patch(color="#f6c1c1",label="manquant"),mp.Patch(color="#e67e22",label="extra")]
    fig.legend(handles=leg,loc="lower center",ncol=4,fontsize=9,bbox_to_anchor=(0.5,-0.06))
    fig.suptitle("G2 — Récupération structurelle (confusion)",y=1.03,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G2_confusion.png",bbox_inches="tight");plt.show()


In [7]:
# === Niveau A — G3 : DAG en graphe de réseau (physique=plein, extra=pointillé, largeur~|A|) ===
POS9={0:(0,0),1:(1,0.6),2:(0,1),3:(1,1.6),4:(0,2),5:(1.6,0.3),6:(1.6,1.3),7:(2.4,0.8),8:(0.8,-1)}
POS11={0:(0,0),1:(1,0.6),2:(0,1),3:(1,1.6),4:(0,2),5:(1.8,0.3),6:(1.8,1.3),7:(2.6,0.8),
       8:(3.2,0.0),9:(3.2,1.6),10:(1.4,-1.2)}
def draw_dag(ax,A,G,labels,pos,title,thr=0.05):
    if not HAS_NX: ax.text(0.5,0.5,"networkx absent",ha="center");ax.set_title(title);return
    A=mask_diag(A);q=A.shape[0]; Gb=np.abs(G)>0
    Dg=nx.DiGraph(); [Dg.add_node(i) for i in range(q)]
    wmax=np.abs(A).max() or 1.0
    for i in range(q):
        for j in range(q):
            if abs(A[i,j])>thr:
                Dg.add_edge(i,j,w=abs(A[i,j]),phys=bool(Gb[i,j]),sign=np.sign(A[i,j]))
    nx.draw_networkx_nodes(Dg,pos,ax=ax,node_color="#dfe6e9",edgecolors="#2d3436",node_size=900)
    nx.draw_networkx_labels(Dg,pos,{i:labels[i] for i in range(q)},ax=ax,font_size=7)
    for (i,j,d) in Dg.edges(data=True):
        ax.annotate("",xy=pos[j],xytext=pos[i],
            arrowprops=dict(arrowstyle="-|>",lw=1+3*d["w"]/wmax,
            color=("#2ecc71" if d["phys"] else "#e67e22"),
            linestyle=("solid" if d["phys"] else "dashed"),alpha=0.85,shrinkA=16,shrinkB=16))
    ax.set_title(title,fontsize=10);ax.axis("off")

graphs=[]
if A9 is not None: graphs.append((A9,G9,VAR_LABELS_9NODE,POS9,"9-node (Oracle)"))
if A11 is not None: graphs.append((A11,G11,VAR_LABELS_V6,POS11,"V6'"))
if graphs:
    n=len(graphs); fig,axes=plt.subplots(1,n,figsize=(6.5*n,6)); axes=np.atleast_1d(axes)
    for ax,(A,G,L,P,T) in zip(axes,graphs): draw_dag(ax,A,G,L,P,T)
    import matplotlib.patches as mp
    fig.legend(handles=[mp.Patch(color="#2ecc71",label="arête physique"),mp.Patch(color="#e67e22",label="arête extra")],
               loc="lower center",ncol=2,bbox_to_anchor=(0.5,-0.03))
    fig.suptitle("G3 — DAG appris (réseau) — largeur ∝ |poids|",y=1.02,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G3_network.png",bbox_inches="tight");plt.show()


In [8]:
# === Niveau A — G4 : force des arêtes (physique vs extra), triées ===
def edge_bars(ax,A,G,labels,title,thr=0.05):
    A=mask_diag(A);q=A.shape[0];Gb=np.abs(G)>0;rows=[]
    for i in range(q):
        for j in range(q):
            if abs(A[i,j])>thr:
                rows.append((f"{labels[i]}→{labels[j]}",abs(A[i,j]),bool(Gb[i,j])))
    rows.sort(key=lambda r:-r[1])
    names=[r[0] for r in rows];vals=[r[1] for r in rows];cols=["#2ecc71" if r[2] else "#e67e22" for r in rows]
    ax.barh(range(len(rows)),vals,color=cols);ax.set_yticks(range(len(rows)));ax.set_yticklabels(names,fontsize=7)
    ax.invert_yaxis();ax.set_title(title,fontsize=10);ax.set_xlabel("|poids A_dag|")

bars=[]
if A9 is not None: bars.append((A9,G9,VAR_LABELS_9NODE,"9-node"))
if A11 is not None: bars.append((A11,G11,VAR_LABELS_V6,"V6'"))
if bars:
    n=len(bars);fig,axes=plt.subplots(1,n,figsize=(6.5*n,6));axes=np.atleast_1d(axes)
    for ax,(A,G,L,T) in zip(axes,bars): edge_bars(ax,A,G,L,T)
    fig.suptitle("G4 — Force des arêtes apprises (vert=physique, orange=extra)",y=1.02,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G4_edge_bars.png",bbox_inches="tight");plt.show()


In [9]:
# === Niveau A — G5 : distribution des poids (signature du verrouillage prior) ===
dists=[]
if A9 is not None: dists.append((A9,"9-node"))
if A11 is not None: dists.append((A11,"V6'"))
if dists:
    fig,axes=plt.subplots(1,len(dists),figsize=(5.5*len(dists),3.6));axes=np.atleast_1d(axes)
    for ax,(A,T) in zip(axes,dists):
        w=np.abs(mask_diag(A)); w=w[w>1e-4]
        ax.hist(w,bins=30,color="#0984e3",alpha=0.8)
        ax.set_title(f"{T} — poids |A_dag| non nuls\n(pic serré = squelette figé/prior-lock)",fontsize=9)
        ax.set_xlabel("|poids|");ax.set_ylabel("compte")
    fig.suptitle("G5 — Distribution des magnitudes d'arêtes",y=1.04,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G5_weight_hist.png",bbox_inches="tight");plt.show()


In [10]:
# === Niveau A — G6 : degrés entrants/sortants + parents de SP_HR (drivers de la précip) ===
def degrees(ax,A,labels,title,thr=0.05):
    A=mask_diag(A);Ab=(np.abs(A)>thr).astype(int)
    out_d=Ab.sum(1);in_d=Ab.sum(0);x=np.arange(len(labels));w=0.4
    ax.bar(x-w/2,out_d,w,label="sortant",color="#6c5ce7")
    ax.bar(x+w/2,in_d,w,label="entrant",color="#00b894")
    ax.set_xticks(x);ax.set_xticklabels(labels,rotation=90,fontsize=7);ax.set_title(title,fontsize=10);ax.legend(fontsize=8)

def sphr_parents(ax,A,labels,title,thr=0.05):
    A=mask_diag(A);j=labels.index("SP_HR");col=np.abs(A[:,j])
    idx=np.argsort(-col);idx=[i for i in idx if col[i]>thr]
    ax.barh([labels[i] for i in idx],[col[i] for i in idx],color="#0984e3")
    ax.invert_yaxis();ax.set_title(title,fontsize=10);ax.set_xlabel("|poids| vers SP_HR")

sets=[]
if A9 is not None: sets.append((A9,VAR_LABELS_9NODE,"9-node"))
if A11 is not None: sets.append((A11,VAR_LABELS_V6,"V6'"))
if sets:
    fig,axes=plt.subplots(2,len(sets),figsize=(6*len(sets),8));axes=np.atleast_2d(axes)
    if axes.shape[0]==1: axes=axes.T
    for k,(A,L,T) in enumerate(sets):
        degrees(axes[0,k],A,L,f"Degrés — {T}")
        sphr_parents(axes[1,k],A,L,f"Parents de SP_HR — {T}")
    fig.suptitle("G6 — Topologie : hubs & drivers de la précipitation (SP_HR)",y=1.01,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G6_degrees.png",bbox_inches="tight");plt.show()


In [11]:
# === Niveau A — G7 : comparaison des métriques d'explicabilité 9-node vs V6' ===
rows=[]
def grab(M,J):
    return dict(q_bin=M.get("q_phys_binary",J.get("q_phys_binary")),
               q_cont=M.get("q_phys_cont",J.get("q_phys_continuous")),
               skelF1=M.get("skeleton_f1",J.get("skeleton_f1")),
               n_extra=M.get("n_extra",J.get("n_extra_edges")),
               phys_mag=J.get("phys_mag_gained"))
data={}
if M9  is not None: data["9-node"]=grab(M9,QJSON9)
if M11 is not None: data["V6'"]=grab(M11,{})
if data:
    keys=["q_bin","q_cont","skelF1","n_extra","phys_mag"]
    fig,axes=plt.subplots(1,len(keys),figsize=(3.2*len(keys),3.4))
    for ax,k in zip(axes,keys):
        names=list(data.keys());vals=[ (data[n].get(k) if data[n].get(k) is not None else 0) for n in names]
        ax.bar(names,vals,color=["#0984e3","#e17055"][:len(names)])
        ax.set_title(k,fontsize=10)
        for i,v in enumerate(vals): ax.text(i,v,f"{v:.2f}" if isinstance(v,float) else str(v),ha="center",va="bottom",fontsize=8)
    fig.suptitle("G7 — Métriques d'explicabilité (↑ mieux sauf n_extra ↓)",y=1.05,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/"G7_metrics.png",bbox_inches="tight");plt.show()
    import pandas as pd
    print(pd.DataFrame(data).T.to_string())


## Niveau B — Prédictions (depuis les caches, sans sampling)

Utilise les caches Stage-1 (`mu_HR`, `baseline_log`, `delta_target`, `valid_mask`) :
- **cible** = `baseline_log + mu_HR + delta_target`
- **prédiction moyenne** = `baseline_log + mu_HR` (le résidu de diffusion est de moyenne ~nulle)

Rapide, pas de GPU. *(Nécessite les fichiers cache — Colab/Drive. En local sans cache, ce niveau est sauté.)*


In [12]:
# === Niveau B — chargement des caches ===
def load_cache(p):
    if not (HAS_TORCH and Path(p).exists()): print("[cache absent]",p); return None
    c=torch.load(p,map_location="cpu",weights_only=False)
    print(f"[cache] {Path(p).name} keys=",list(c.keys()))
    return c
C9  = load_cache(CACHE_9)
CV6 = load_cache(CACHE_V6)

def fields(c):
    if c is None: return None
    def g(k):
        v=c.get(k);
        return v.float().numpy() if (v is not None and hasattr(v,"numpy")) else (np.asarray(v) if v is not None else None)
    mu=g("mu_HR");bl=g("baseline_log");dt=g("delta_target");vm=g("valid_mask")
    if mu is None or bl is None or dt is None:
        print("  [warn] clefs manquantes"); return None
    tgt=bl+mu+dt; mean=bl+mu
    if vm is not None:
        m=vm>0.5; tgt=np.where(m,tgt,np.nan); mean=np.where(m,mean,np.nan)
    return dict(target=tgt,mean=mean,mu=mu,baseline=bl,delta=dt,mask=vm)

F9  = fields(C9)
FV6 = fields(CV6)
print("F9:",None if F9 is None else F9["target"].shape,"| FV6:",None if FV6 is None else FV6["target"].shape)


[cache absent] C:\Users\reall\Desktop\climate_data\oracle_9node\seed_42\stage1_cache.pt
[cache absent] C:\Users\reall\Desktop\climate_data\oracle_v6_prime\seed_42\bs32b_cache_seed42.pt
F9: None | FV6: None


In [13]:
# === Niveau B — G8 : cartes cible / prédiction moyenne / erreur ===
def show_maps(F,title,nsamp=3):
    if F is None: print("[skip]",title,"(pas de cache)"); return
    N=F["target"].shape[0]; idx=np.linspace(0,N-1,min(nsamp,N)).astype(int)
    fig,axes=plt.subplots(len(idx),3,figsize=(11,3.4*len(idx)));axes=np.atleast_2d(axes)
    for r,s in enumerate(idx):
        t=F["target"][s,0]; m=F["mean"][s,0]; e=m-t
        for c,(img,ttl,cm) in enumerate([(t,"cible",  "viridis"),(m,"préd. moyenne","viridis"),(e,"erreur","RdBu_r")]):
            v=np.nanmax(np.abs(img)) or 1
            im=axes[r,c].imshow(img,cmap=cm,vmin=(-v if c==2 else None),vmax=(v if c==2 else None),origin="lower")
            axes[r,c].set_title(f"{ttl} #{s}",fontsize=9);axes[r,c].axis("off");plt.colorbar(im,ax=axes[r,c],fraction=0.046)
    fig.suptitle(f"G8 — {title} (espace log)",y=1.01,fontsize=12)
    plt.tight_layout();plt.savefig(FIG_DIR/f"G8_maps_{title.replace(chr(39),'').replace(' ','_')}.png",bbox_inches="tight");plt.show()

show_maps(F9,"9-node")
show_maps(FV6,"V6'")


[skip] 9-node (pas de cache)
[skip] V6' (pas de cache)


In [14]:
# === Niveau B — G9 : scatter préd vs cible + histogramme des extrêmes ===
def scatter_hist(F,title):
    if F is None: print("[skip]",title); return
    t=F["target"].ravel(); m=F["mean"].ravel(); ok=~(np.isnan(t)|np.isnan(m))
    t,m=t[ok],m[ok];
    if t.size>200000:
        s=np.random.default_rng(0).choice(t.size,200000,replace=False); t,m=t[s],m[s]
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].hexbin(t,m,gridsize=60,cmap="viridis",bins="log");lim=[min(t.min(),m.min()),max(t.max(),m.max())]
    ax[0].plot(lim,lim,"r--",lw=1);ax[0].set_xlabel("cible");ax[0].set_ylabel("préd moyenne")
    r=np.corrcoef(t,m)[0,1];ax[0].set_title(f"{title} — préd vs cible (r={r:.3f})",fontsize=10)
    p=[90,95,99,99.9]
    ax[1].plot(p,[np.percentile(t,q) for q in p],"o-",label="cible")
    ax[1].plot(p,[np.percentile(m,q) for q in p],"s--",label="préd moyenne")
    ax[1].set_xlabel("percentile");ax[1].set_ylabel("valeur (log)");ax[1].legend();ax[1].set_title(f"{title} — queue",fontsize=10)
    plt.tight_layout();plt.savefig(FIG_DIR/f"G9_scatter_{title.replace(chr(39),'').replace(' ','_')}.png",bbox_inches="tight");plt.show()

scatter_hist(F9,"9-node")
scatter_hist(FV6,"V6'")


[skip] 9-node
[skip] V6'


In [15]:
# === Niveau B — G10 : cartes de biais moyen et RMSE par pixel ===
def err_maps(F,title):
    if F is None: print("[skip]",title); return
    err=F["mean"]-F["target"]  # (N,1,H,W)
    bias=np.nanmean(err,axis=0)[0]; rmse=np.sqrt(np.nanmean(err**2,axis=0))[0]
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    v=np.nanmax(np.abs(bias)) or 1
    im0=ax[0].imshow(bias,cmap="RdBu_r",vmin=-v,vmax=v,origin="lower");ax[0].set_title(f"{title} — biais moyen",fontsize=10);plt.colorbar(im0,ax=ax[0],fraction=0.046)
    im1=ax[1].imshow(rmse,cmap="magma",origin="lower");ax[1].set_title(f"{title} — RMSE/pixel",fontsize=10);plt.colorbar(im1,ax=ax[1],fraction=0.046)
    for a in ax: a.axis("off")
    plt.tight_layout();plt.savefig(FIG_DIR/f"G10_err_{title.replace(chr(39),'').replace(' ','_')}.png",bbox_inches="tight");plt.show()

err_maps(F9,"9-node")
err_maps(FV6,"V6'")


[skip] 9-node
[skip] V6'


## Niveau C — (optionnel) Sampling complet + ablation A_dag

Recharge le stack complet et échantillonne la diffusion. Mesure l'**ablation A_dag** (mettre A_dag=0 et regarder Δ/signal) — le vrai test « le DAG conditionne-t-il la sortie ? ».

⚠️ **Colab + checkpoints Drive requis, et parité d'environnement avec le notebook d'entraînement.** Mets `RUN_TIER_C = True` pour l'activer. Laisse `False` si tu veux juste A+B.

Le code d'ablation est copié de la logique Figure 5 de `st_cdgm_v5_evaluation.ipynb` :
```python
A_orig = rcn_cell.A_dag.detach().clone()
# ... predict full ...
rcn_cell.A_dag.data.zero_()          # ablation
# ... predict ablated ...
rcn_cell.A_dag.data.copy_(A_orig)    # restore
delta_signal_ratio = |Δ|.mean() / |mu_full|.mean()
```
Pour reconstruire le stack, réutilise `build_fresh_stack(seed)` du notebook 9-node (ou le stack factory V6′) puis charge les `*_state_dict`. Ne PAS réimplémenter à la main (fragile).


In [16]:
# === Niveau C — scaffold (désactivé par défaut) ===
RUN_TIER_C = False   # <-- passe à True sur Colab avec le stack déjà construit

if RUN_TIER_C:
    # Attendu dans le namespace : un stack dict + predict_with_stack (copie du notebook source).
    # stack = build_fresh_stack(42); charger les state_dicts depuis CKPT_9 ...
    def ablation_delta(stack, batch, predict_fn, K=1, n_steps=18):
        rcn=stack["rcn_cell"]; A0=rcn.A_dag.detach().clone()
        full=predict_fn(stack,batch,K=K,n_steps=n_steps).nanmean(0).squeeze().cpu().numpy()
        rcn.A_dag.data.zero_()
        abl =predict_fn(stack,batch,K=K,n_steps=n_steps).nanmean(0).squeeze().cpu().numpy()
        rcn.A_dag.data.copy_(A0)
        delta=abl-full; ratio=float(np.abs(delta).mean()/max(np.abs(full).mean(),1e-12))
        fig,ax=plt.subplots(1,3,figsize=(12,4))
        for a,(img,t,cm) in zip(ax,[(full,"A_dag appris","viridis"),(abl,"A_dag=0","viridis"),(delta,f"Δ (Δ/signal={ratio:.1%})","RdBu_r")]):
            v=np.nanmax(np.abs(img)) or 1;im=a.imshow(img,cmap=cm,vmin=(-v if cm=="RdBu_r" else None),vmax=(v if cm=="RdBu_r" else None),origin="lower")
            a.set_title(t,fontsize=10);a.axis("off");plt.colorbar(im,ax=a,fraction=0.046)
        plt.tight_layout();plt.savefig(FIG_DIR/"GC_ablation.png",bbox_inches="tight");plt.show()
        return ratio
    print("Niveau C prêt : appelle ablation_delta(stack, batch, predict_with_stack).")
else:
    print("Niveau C désactivé (RUN_TIER_C=False). Niveaux A et B suffisent pour l'explicabilité structurelle + prédictions.")


Niveau C désactivé (RUN_TIER_C=False). Niveaux A et B suffisent pour l'explicabilité structurelle + prédictions.


## Lecture des résultats

- **G1–G7 (structurel)** : si `skeleton_f1≈1`, `n_extra≈0` et les arêtes vertes dominent → le DAG **matche la physique**. Si les poids sont ~uniformes (G5, pic serré) → **verrouillage sur le prior** (imposé, pas découvert).
- **G8–G10 (prédictions)** : qualité du champ moyen, biais, queue. Le résidu stochastique n'y est pas (moyenne nulle) — voir Niveau C pour le sampling.
- **Niveau C (ablation)** : `Δ/signal` faible (<~10%) → le décodeur **ignore largement A_dag** (conditionnement décoratif). C'est le test décisif d'explicabilité *fonctionnelle*.

> Rappel honnête : structure **fidèle et auditable** ≠ mécanisme causal **à fort levier**. Revendiquer « physique imposée », pas « découverte ».
